# Zero-shot top-down vs bottom-up exploration

This notebook loads the ISCO multilingual test sentences, derives the inference
text for each sample, and calls the same Kedro node functions used in production
pipelines to compute the top-down routing and bottom-up validation routes.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import yaml

from taxomind.pipelines.zero_shot import nodes as zero_nodes

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / "data"
CONF_DIR = PROJECT_DIR / "conf"

print(f"Project directory: {PROJECT_DIR}")


In [ ]:
test_path = DATA_DIR / "01_raw" / "isco_test_sentences.json"
taxonomy_path = DATA_DIR / "05_model_input" / "taxonomy_embedded.parquet"
parameters_path = CONF_DIR / "base" / "parameters.yml"

with test_path.open(encoding="utf-8") as fp:
    test_payload = json.load(fp)
taxonomy_df = pd.read_parquet(taxonomy_path)
with parameters_path.open(encoding="utf-8") as fp:
    parameters = yaml.safe_load(fp)

model_name = parameters["zero_shot"]["model_name"]
print(f"Loaded {len(test_payload.get('sentences', []))} test sentences.")
print(f"Taxonomy rows: {len(taxonomy_df):,}")
print(f"Embedding model: {model_name}")


In [ ]:
sentences = []
for record in test_payload.get("sentences", []):
    inference_text = zero_nodes.compose_inference_text(record.get("fields", {}))
    sentences.append(
        {
            "sentence_id": record.get("sentence_id"),
            "taxonomyKey": test_payload.get("taxonomyKey"),
            "text": inference_text,
        }
    )

sentences_df = pd.DataFrame(sentences)
sentences_df

In [ ]:
results = []
for row in sentences_df.itertuples(index=False):
    topdown = zero_nodes.top_down_route(row.text, taxonomy_df, model_name)
    bottomup = zero_nodes.bottom_up_validation(row.text, taxonomy_df, topdown, model_name)
    results.append(
        {
            "sentence_id": row.sentence_id,
            "text": row.text,
            "top_down_route": topdown.get("route", []),
            "bottom_up_route": bottomup.get("route", []),
            "routes_match": bottomup.get("routes_match"),
        }
    )

print(f"Computed routes for {len(results)} sentences.")


In [ ]:
example = results[0]
print(f"Sentence ID: {example['sentence_id']}")
print(example["text"])
pd.DataFrame(example["top_down_route"])


In [ ]:
pd.DataFrame(example["bottom_up_route"])